# 04 - 硬约束边界条件

上一节 Poisson 方程通过边界损失让 `u(0)=u(1)=0`。本节换一种方式：直接构造

`u(x) = x(1-x) N_theta(x)`

因为 `x(1-x)` 在 `x=0` 和 `x=1` 都为 0，所以边界条件天然满足。

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(4)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
class MLP(nn.Module):
    def __init__(self, width=48, depth=4):
        super().__init__()
        layers = [nn.Linear(1, width), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.Tanh()]
        layers += [nn.Linear(width, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

base_model = MLP().to(device)

def u_model(x):
    return x * (1 - x) * base_model(x)

def exact_solution(x):
    return torch.sin(math.pi * x)

In [ ]:
def residual(x):
    x.requires_grad_(True)
    u = u_model(x)
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    forcing = -(math.pi**2) * torch.sin(math.pi * x)
    return u_xx - forcing

optimizer = torch.optim.Adam(base_model.parameters(), lr=1e-3)
history = []

for epoch in range(6000):
    x_f = torch.rand(256, 1, device=device)
    loss = torch.mean(residual(x_f)**2)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    history.append(loss.item())
    if epoch % 1000 == 0:
        print(f"epoch={epoch:4d}, physics loss={loss.item():.3e}")

In [ ]:
x = torch.linspace(0, 1, 400, device=device).reshape(-1, 1)
with torch.no_grad():
    pred = u_model(x)
    exact = exact_solution(x)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(x.cpu(), exact.cpu(), label="exact")
plt.plot(x.cpu(), pred.cpu(), "--", label="hard-BC PINN")
plt.legend()
plt.subplot(1, 2, 2)
plt.semilogy(history)
plt.title("Physics loss only")
plt.tight_layout()
plt.show()

print("u(0)=", u_model(torch.zeros(1, 1, device=device)).item())
print("u(1)=", u_model(torch.ones(1, 1, device=device)).item())
print("max abs error=", torch.max(torch.abs(pred - exact)).item())

练习：把边界改成 `u(0)=1, u(1)=2`。提示：可构造 `u(x)=1+x+x(1-x)N(x)`。